[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/metaflow-certified/notebooks/day-02-flow-architecture.ipynb#scrollTo=1a2b3c4d)

---
# Day 2 · Flow Architecture and DAG Fundamentals
**certified-journeys / metaflow-certified** · Day 2 · Learn

> **Goal for today:** Design a multi-step ML pipeline with 4+ linear steps, correctly apply the `@step` decorator, pass data between steps via `self` attributes, and visualize the flow graph structure.

In [ ]:
%pip install -q metaflow

## Step 1 · Understanding DAGs in Metaflow

A **Directed Acyclic Graph (DAG)** defines the order of operations in a Metaflow pipeline.

- **Directed** — edges have direction (data flows one way)
- **Acyclic** — no cycles (no step can call itself, directly or indirectly)
- **Graph** — multiple paths are possible (fan-out / fan-in)

Metaflow builds the DAG by introspecting your `self.next()` calls at import time:

```
Linear flow:      start → load_data → preprocess → train → evaluate → end

Fan-out / Fan-in: start → load_data → [train_modelA, train_modelB] → compare → end
                                            ↑ runs in parallel ↑
```

Today we focus on **linear flows** — the foundation before we add parallelism.

| DAG element | Metaflow construct |
|---|---|
| Node (step) | Method decorated with `@step` |
| Edge (transition) | `self.next(self.step_name)` |
| Entry point | Step named `start` |
| Exit point | Step named `end` |
| Data on edge | `self.attribute` — serialized automatically |

In [ ]:
%%writefile pipeline_flow.py
from metaflow import FlowSpec, step
import json

class MLPipelineFlow(FlowSpec):
    """
    A realistic 5-step linear ML pipeline demonstrating:
    - Data loading and validation
    - Preprocessing
    - Model training (simulated)
    - Evaluation
    """

    @step
    def start(self):
        # Pipeline initialization — define config and validate inputs
        self.config = {
            "dataset": "iris_sample",
            "test_split": 0.2,
            "random_seed": 42,
            "model_type": "logistic_regression"
        }
        print(f"Starting pipeline with config: {json.dumps(self.config, indent=2)}")
        self.next(self.load_data)

    @step
    def load_data(self):
        # Load data — in production this might read from S3 or a database
        # Here we generate a synthetic dataset to avoid external dependencies
        import random
        random.seed(self.config["random_seed"])

        # Simulate loading 150 samples with 4 features (like Iris)
        n_samples = 150
        self.raw_data = [
            {
                "sepal_length": round(random.uniform(4.3, 7.9), 1),
                "sepal_width":  round(random.uniform(2.0, 4.4), 1),
                "petal_length": round(random.uniform(1.0, 6.9), 1),
                "petal_width":  round(random.uniform(0.1, 2.5), 1),
                "label":        random.choice([0, 1, 2])  # 3 classes
            }
            for _ in range(n_samples)
        ]
        self.n_samples = len(self.raw_data)
        self.n_features = 4
        print(f"Loaded {self.n_samples} samples with {self.n_features} features")
        self.next(self.preprocess)

    @step
    def preprocess(self):
        # Preprocess: feature extraction and train/test split
        import random
        random.seed(self.config["random_seed"])

        feature_keys = ["sepal_length", "sepal_width", "petal_length", "petal_width"]

        # Extract features (X) and labels (y)
        X = [[row[k] for k in feature_keys] for row in self.raw_data]
        y = [row["label"] for row in self.raw_data]

        # Manual min-max normalization per feature
        for col in range(self.n_features):
            col_vals = [X[i][col] for i in range(len(X))]
            col_min, col_max = min(col_vals), max(col_vals)
            span = col_max - col_min or 1e-9  # avoid division by zero
            for i in range(len(X)):
                X[i][col] = round((X[i][col] - col_min) / span, 4)

        # Train/test split
        indices = list(range(len(X)))
        random.shuffle(indices)
        split_idx = int(len(indices) * (1 - self.config["test_split"]))
        train_idx = indices[:split_idx]
        test_idx  = indices[split_idx:]

        self.X_train = [X[i] for i in train_idx]
        self.y_train = [y[i] for i in train_idx]
        self.X_test  = [X[i] for i in test_idx]
        self.y_test  = [y[i] for i in test_idx]

        print(f"Preprocessed: {len(self.X_train)} train, {len(self.X_test)} test samples")
        self.next(self.train)

    @step
    def train(self):
        # Train a model — we use a simple majority-class classifier to avoid sklearn dependency
        # Production equivalent: use sklearn, XGBoost, or PyTorch here
        from collections import Counter

        # Count class frequencies in training data
        label_counts = Counter(self.y_train)
        # The "model" predicts the most common class (baseline)
        majority_class = label_counts.most_common(1)[0][0]

        # Store the "trained model" as a serializable dict
        self.model = {
            "type": "majority_class_baseline",
            "majority_class": majority_class,
            "class_counts": dict(label_counts),
            "n_train_samples": len(self.y_train)
        }
        print(f"Trained model: {self.model}")
        self.next(self.evaluate)

    @step
    def evaluate(self):
        # Evaluate the model on test data
        from collections import Counter

        # Generate predictions (always predict majority class)
        predictions = [self.model["majority_class"]] * len(self.y_test)

        # Calculate accuracy
        correct = sum(p == t for p, t in zip(predictions, self.y_test))
        self.accuracy = round(correct / len(self.y_test), 4)

        # Per-class counts
        self.test_distribution = dict(Counter(self.y_test))

        print(f"Test accuracy: {self.accuracy:.2%}")
        print(f"Test class distribution: {self.test_distribution}")
        self.next(self.end)

    @step
    def end(self):
        # Summarize the pipeline run
        print("=" * 50)
        print("Pipeline complete!")
        print(f"  Dataset:      {self.config['dataset']}")
        print(f"  Samples:      {self.n_samples}")
        print(f"  Model:        {self.model['type']}")
        print(f"  Test accuracy: {self.accuracy:.2%}")
        print("=" * 50)

if __name__ == '__main__':
    MLPipelineFlow()

**What just happened?**

- We defined a **5-step linear flow**: `start → load_data → preprocess → train → evaluate → end`.
- Each step stores data on `self` — for example, `preprocess` reads `self.raw_data` (set by `load_data`) and writes `self.X_train`, `self.y_train`, etc.
- **No sklearn required** — the "model" is a simple majority-class baseline; in production you'd swap in a real estimator.
- The `@step` decorator and `self.next()` are the only Metaflow-specific pieces — the rest is plain Python.

## Step 2 · Running the Multi-Step Pipeline

When you run a flow, Metaflow:
1. Validates the DAG (checks all `self.next()` calls resolve to valid steps)
2. Runs each step **in a separate OS process** for isolation
3. **Serializes** all `self.*` attributes between steps
4. Stores metadata (start/end time, status, parameters) in the datastore

The `--no-pylint` flag suppresses the linting pass (useful in Colab where pylint may not be configured).

In [ ]:
# Run the 5-step pipeline
!python pipeline_flow.py run --no-pylint 2>&1

**What just happened?**

- Metaflow ran all 5 steps in sequence, printing each step's output prefixed with `[timestamp]`.
- **Each step shows its own process ID (pid)** — confirming true process isolation.
- **If any step had failed**, Metaflow would stop and show a clear error. You could then fix the code and run `python pipeline_flow.py resume` to restart from the failed step.
- The `accuracy` artifact is now stored — we can query it from the Client API without re-running.

## Step 3 · Visualizing the Flow Graph

Metaflow's `show` command prints the DAG structure without executing any steps.
It's fast — useful for verifying the graph before a long run.

```bash
python flow.py show          # print DAG to stdout
python flow.py show --graph  # also available in some versions
```

For richer visualization, **Metaflow UI** (open-source web app) shows:
- Flow DAG with step status colors
- Artifact sizes and types per step
- Log streaming for long-running steps
- Comparison across multiple runs

In [ ]:
# Visualize the flow graph — shows DAG without running any code
!python pipeline_flow.py show 2>&1

In [ ]:
# We can also build a simple ASCII DAG visualization using the Client API
from metaflow import Flow

run = Flow("MLPipelineFlow").latest_run
print(f"Run ID: {run.id}  |  Successful: {run.successful}")
print()
print("Step execution order (DAG nodes):")
print()

# Collect step info
steps_info = []
for step in run:
    for task in step:
        steps_info.append({
            "step": step.id,
            "task": task.id,
            "created": task.created_at,
            "finished": task.finished_at,
            "ok": task.successful
        })

# Sort by creation time to reconstruct order
steps_info.sort(key=lambda x: x["created"])

# Print as a linear DAG
for i, s in enumerate(steps_info):
    status = "✓" if s["ok"] else "✗"
    arrow = " → " if i < len(steps_info) - 1 else ""
    print(f"[{status}] {s['step']}{arrow}", end="")
print()
print()

# Access the accuracy artifact from the evaluate step
evaluate_task = next(iter(run["evaluate"]))
print(f"Accuracy artifact: {evaluate_task.data.accuracy:.2%}")
print(f"Model artifact: {evaluate_task.data.model}")

**What just happened?**

- `python flow.py show` printed the DAG **without executing** — great for CI checks.
- The Client API lets us **reconstruct execution order** by querying step creation times.
- **`run["evaluate"]`** is step-level access — we get only the evaluate step's artifacts without loading everything.
- `task.data.accuracy` deserializes the stored float — available from any Python session.

## Step 4 · Passing Data Between Steps: The Self-Attribute Contract

Understanding **what persists** between steps is critical:

| Survives step boundary? | How |
|---|---|
| `self.attribute` assigned in step N | ✅ Yes — serialized to datastore |
| Local variable `x = 5` in step N | ❌ No — dies with the process |
| Module-level global in step N | ❌ No — fresh process, globals re-initialized |
| `self.attribute` from step N-2 | ✅ Yes — Metaflow merges artifacts transitively |

**Serialization rules:**
- Most Python objects: serialized with `pickle`
- `pandas.DataFrame`, `numpy.ndarray`: custom efficient serializers
- Large objects (>1MB): stored as separate files, still transparent
- Un-picklable objects (file handles, sockets): will raise a clear error

In [ ]:
%%writefile data_flow_demo.py
from metaflow import FlowSpec, step

class DataFlowDemo(FlowSpec):
    """Demonstrates how self attributes flow through steps."""

    @step
    def start(self):
        # Set several artifact types to demonstrate serialization
        self.a_string = "Hello, I am a string artifact"
        self.a_number = 3.14159
        self.a_list   = [10, 20, 30, 40, 50]
        self.a_dict   = {"model": "lr", "C": 1.0, "max_iter": 100}
        self.a_set    = {"train", "test", "val"}  # sets are picklable

        # This local variable will NOT be available in the next step
        local_var = "I only live in this step's process"
        print(f"local_var (only visible here): {local_var!r}")
        print(f"self.a_string will survive to next step: {self.a_string!r}")
        self.next(self.step_two)

    @step
    def step_two(self):
        # All self.* from start are available here
        # local_var from start is gone — can't access it
        print("In step_two — reading artifacts set in start:")
        print(f"  self.a_string = {self.a_string!r}")
        print(f"  self.a_number = {self.a_number}")
        print(f"  self.a_list   = {self.a_list}")
        print(f"  self.a_dict   = {self.a_dict}")
        print(f"  self.a_set    = {self.a_set}")

        # Add a new artifact — it will be available in end
        self.list_sum = sum(self.a_list)
        print(f"  Computed self.list_sum = {self.list_sum}")
        self.next(self.step_three)

    @step
    def step_three(self):
        # Artifacts from BOTH start AND step_two are available here
        print("In step_three — all previous artifacts available:")
        print(f"  self.a_string from start:    {self.a_string!r}")
        print(f"  self.list_sum from step_two: {self.list_sum}")

        # Modify an inherited artifact — creates a new version in this step's namespace
        self.a_list = self.a_list + [60, 70]  # doesn't mutate start's artifact
        print(f"  self.a_list (extended):       {self.a_list}")
        self.next(self.end)

    @step
    def end(self):
        print("Final artifact summary:")
        print(f"  self.a_list has {len(self.a_list)} elements (extended in step_three)")
        print(f"  self.list_sum = {self.list_sum}")
        print("All artifacts are now versioned in the datastore.")

if __name__ == '__main__':
    DataFlowDemo()

In [ ]:
# Run the data flow demo
!python data_flow_demo.py run --no-pylint 2>&1

**What just happened?**

- We confirmed that **all `self.*` attributes** — strings, numbers, lists, dicts, sets — are automatically serialized between steps.
- **Local variables are not preserved** — only `self.attribute` assignments survive step boundaries.
- **Artifact inheritance is transitive** — `step_three` can read `self.a_string` even though it was set two steps earlier.
- **Reassigning a `self` attribute** creates a new version in that step's namespace — the original from `start` is still accessible via the Client API under the `start` step.

## Step 5 · Comparing Runs with the Client API

Because every run has a unique ID and all artifacts are versioned, you can compare runs:

```python
Flow('MyFlow').runs()          # iterator over all runs
Flow('MyFlow')['42']           # specific run by ID
Flow('MyFlow').latest_run      # most recent run
Flow('MyFlow').latest_successful_run  # most recent successful run
```

This is the foundation of **experiment tracking** in Metaflow — no external MLflow/W&B needed for basic comparisons.

In [ ]:
from metaflow import Flow

# List all runs of MLPipelineFlow and compare their accuracy artifacts
flow = Flow("MLPipelineFlow")

print("Run history for MLPipelineFlow:")
print(f"{'Run ID':>8}  {'Successful':>10}  {'Accuracy':>10}  {'Samples':>8}")
print("-" * 45)

for run in flow.runs():
    try:
        evaluate_task = next(iter(run["evaluate"]))
        accuracy = evaluate_task.data.accuracy
        n_samples = evaluate_task.data.n_samples
        print(f"{run.id:>8}  {str(run.successful):>10}  {accuracy:>10.2%}  {n_samples:>8}")
    except Exception:
        # Handle runs that might not have these artifacts
        print(f"{run.id:>8}  {str(run.successful):>10}  {'N/A':>10}  {'N/A':>8}")

**What just happened?**

- `Flow('MLPipelineFlow').runs()` returns **all historical runs** — not just the latest.
- We extracted `accuracy` and `n_samples` from each run's evaluate step without re-running anything.
- **This is the foundation of experiment tracking** — run your pipeline with different configs and compare artifacts programmatically.
- `run.successful` filters out failed runs — useful when searching for the best model across experiments.

In [ ]:
# Challenge: Build a 4-step flow called FeatureEngineeringFlow that:
# Step 1 (start):   generates self.raw_numbers = list of 20 random integers between 1 and 100
# Step 2 (compute): calculates self.mean, self.std (standard deviation), self.min, self.max
# Step 3 (normalize): creates self.normalized = [(x - mean) / std for each x in raw_numbers]
# Step 4 (end):     prints a summary table of all computed statistics
#
# Bonus: After running, use the Client API to:
#   - Print the normalized list from the 'normalize' step
#   - Verify that the mean of the normalized list is approximately 0

%%writefile challenge_flow.py
from metaflow import FlowSpec, step
import random
import math

class FeatureEngineeringFlow(FlowSpec):

    @step
    def start(self):
        # TODO: generate self.raw_numbers = 20 random integers between 1 and 100
        random.seed(99)
        ...
        self.next(self.compute)

    @step
    def compute(self):
        # TODO: calculate self.mean, self.std, self.min, self.max
        ...
        self.next(self.normalize)

    @step
    def normalize(self):
        # TODO: create self.normalized using z-score normalization
        ...
        self.next(self.end)

    @step
    def end(self):
        # TODO: print a summary of all statistics
        ...

if __name__ == '__main__':
    FeatureEngineeringFlow()

# Run with: !python challenge_flow.py run --no-pylint
# Then use the Client API to inspect the normalized list

---
## Day 2 key concepts recap

| Concept | What to remember |
|---|---|
| DAG | Directed Acyclic Graph — nodes are `@step` methods, edges are `self.next()` calls |
| `@step` placement | Every step method needs the decorator — forgetting it silently breaks the DAG |
| `self.next()` | Must be the **last call** in every step (except `end`) |
| Artifact scope | Only `self.attribute` survives step boundaries — local vars die with the process |
| Transitive inheritance | Artifacts flow forward through the DAG — `step N+2` can read what `step N` set |
| `python flow.py show` | Validates and prints the DAG without executing any code |
| Run comparison | `Flow('Name').runs()` iterates all runs for experiment comparison |

> **Tip:** Every step in Metaflow is a checkpoint — if a step fails, you can resume from where you left off.

---
## What's next
**Day 3** → Data Artifacts and the Metaflow Store — deep-dive into artifact versioning, the Client API for experiment tracking, and exploring the `.metaflow` directory structure.

Mark Day 2 complete in your [tracker](../index.html).